# 02 — Agent traces

**Acceptance gate C2**: *les 5 formes de collaboration sont démontrables* — a LangGraph
trace showing handoff, delegation, loop, vote and interrupt.

This notebook does not simulate anything. It reads `data/checkpoints.sqlite` — the same
SQLite checkpointer the running system writes to — and reconstructs a real run that was
driven through the browser on 2026-08-03 against a live model and the Docker sandbox.

Every cell is offline. Nothing here needs an API key or a network.

In [ ]:
import json
import sqlite3
from collections import Counter
from pathlib import Path

DB = Path("../data/checkpoints.sqlite")
if not DB.exists():
    DB = Path("data/checkpoints.sqlite")      # when run from the repo root

conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row
print(f"checkpoint store: {DB.resolve()}")

threads = [r[0] for r in conn.execute("SELECT DISTINCT thread_id FROM checkpoints")]
print(f"{len(threads)} sessions on record")

## 1. Picking the run

We want the richest session — the one that reached the reviewer and looped. Rank the
threads by how many distinct graph nodes they entered.

In [ ]:
def node_sequence(thread_id):
    """The nodes a run entered, in order.

    LangGraph records a `branch:to:<node>` write when control moves to that node, so the
    node sequence is recoverable from the checkpoint alone — no separate trace log.
    """
    rows = conn.execute(
        "SELECT channel FROM writes WHERE thread_id = ? ORDER BY rowid", (thread_id,)
    )
    seq = []
    for (channel,) in rows:
        if channel.startswith("branch:to:"):
            seq.append(channel.removeprefix("branch:to:"))
        elif channel in ("__interrupt__", "__resume__", "__error__"):
            seq.append(channel)
    return seq


ranked = sorted(threads, key=lambda t: len(set(node_sequence(t))), reverse=True)
for t in ranked[:5]:
    print(f"{t}: {len(set(node_sequence(t))):2d} distinct nodes")

RUN = ranked[0]
print(f"\nusing {RUN}")

In [ ]:
seq = node_sequence(RUN)
print("  " + "\n→ ".join(seq))

## 2. The five forms of collaboration

| # | Form | Where it is in this trace | Mechanism in code |
|---|---|---|---|
| 1 | **Handoff** | `retriever → planner` | a typed `ContextPack` crosses the edge; the planner never re-retrieves |
| 2 | **Delegation** | `planner → regression → editor` | the plan's steps are the work order the editor executes |
| 3 | **Loop** | `reviewer → editor` (second pass) | `Command(goto="editor")` on a `revise` verdict, capped by `iterations` |
| 4 | **Vote / contested** | `contested` counter, `escalation` node | editor↔reviewer disagreement per file; escalates to a human instead of looping forever |
| 5 | **Interrupt** | `__interrupt__` / `__resume__` | `interrupt()` persists the state and stops; `/approve` resumes with `Command(resume=…)` |

The next cells show each one in the recorded data.

In [ ]:
# --- form 5: interrupt. Both §5.5 gates, and what they asked the human. ---
gates = [n for n in seq if n in ("plan_approval", "patch_approval", "__interrupt__", "__resume__")]
print("gate-related events:", gates)

interrupts = conn.execute(
    "SELECT checkpoint_id FROM writes WHERE thread_id = ? AND channel = '__interrupt__'", (RUN,)
).fetchall()
print(f"\n{len(interrupts)} interrupt(s) — the run stopped and waited for a person")
print("an approval therefore survives a closed tab, an hour, or a process restart")

In [ ]:
# --- form 3: the repair loop. Count how often each node was entered. ---
counts = Counter(seq)
looped = {n: c for n, c in counts.items() for _ in [0] if c > 1 and not n.startswith("__")}
print("nodes entered more than once:", looped or "none")
print()
passes = counts.get("editor", 0)
print(f"editor passes: {passes}  ->  {max(0, passes - 1)} repair iteration(s)")
print()
print("Every pass after the first is the loop: the reviewer returned `revise`, and the")
print("editor rewrote the patch after seeing what the sandbox actually reported.")
print("`verify` was entered", counts.get("verify", 0), "times — each patch was re-tested,")
print("not re-argued. The exit code is the authority, not the reviewer's opinion.")

### What the loop produced

The first patch was rejected by the sandbox, not by an opinion — `453 passed, 28 failed`.
The reviewer returned `revise`, and the editor's second attempt widened the guard:

```diff
-    if not isinstance(sql, str):
-        raise TypeError('Expected string, got {!r}'.format(type(sql)))
+    if not isinstance(sql, (str, bytes, TextIOBase)) and not hasattr(sql, 'read'):
+        raise TypeError('Expected string or file-like object, got {!r}'.format(type(sql)))
```

Nobody told it that `sqlparse` accepts byte strings and file objects. It read the
failures. That is the collaboration form worth defending: the reviewer and the sandbox
together carry information the editor did not have on its first pass.

In [ ]:
# --- the guardrails that ran during this session (cahier §8.5) ---
events = conn.execute(
    "SELECT stage, rule, action, target, detail FROM guardrail_events "
    "WHERE session_id = ? ORDER BY rowid",
    (RUN,),
).fetchall()

print(f"{len(events)} guardrail events for this run\n")
for (action, rule), n in Counter((e["action"], e["rule"]) for e in events).most_common():
    print(f"  {n:4d}  [{action}] {rule}")

In [ ]:
# --- the one that fired: indirect injection on retrieved code (§8.2) ---
fired = [e for e in events if e["action"] != "allowed"]
for e in fired:
    print(f"[{e['action'].upper()}] {e['rule']}  ({e['stage']})")
    print(f"   target : {e['target']}")
    print(f"   detail : {e['detail']}")

if not fired:
    print("no non-allowed events in this session")
else:
    print("\nA comment reading 'Ignore all previous instructions…' was planted in the")
    print("target repo. It was redacted from the pack BEFORE the planner saw it, and the")
    print("model never acted on it. `allowed` events are logged just as deliberately —")
    print("a log that only records refusals cannot show that a check ran on a clean run.")

## 3. What this trace does and does not prove

**Proves.** Six distinct agents, each with its own typed payload. Both human gates
actually stopping the run. A repair loop that improved a patch using sandbox output. An
indirect-injection defence firing on real third-party text. All of it reconstructed from
the checkpoint store, so it is the system's own record rather than a transcript.

**Does not prove.** The run ended on `__error__` — the free-tier daily quota (`429`)
expired mid-repair, so the re-verification after the second patch never ran. The red half
of red→green is recorded (`exit=1, 0 passed, 1 failed`); the green half is not.

**Form 4 (vote/contested) is wired but did not fire here.** `make_escalation_node`
escalates to a human when the editor and reviewer keep disagreeing about the same file;
this run agreed on the second pass, so the counter never reached its threshold. The
mechanism is covered by `tests/test_review_hitl.py`.